# Script 12 - Rest-activity rhythm dimension specificity (Supplementary Table S9 + Supp Fig S2)

Fits each of the three non-parametric rest-activity rhythm dimensions against
each outcome, so the association can be compared across dimensions:

- **IV (intradaily variability)** — hour-to-hour fragmentation of the daily
  activity profile.
- **IS (interdaily stability)** — across-day regularity of the activity
  profile.
- **RA (relative amplitude)** = $(M_{10} - L_5) / (M_{10} + L_5)$ — strength of
  the diurnal cycle.

All three dimensions are derived in `scripts/5_accelerometer_RAR.ipynb`, with
IV and IS on a common hourly epoch and RA at minute resolution from M10 and L5.
Whether a dimension is treated as an inferential exposure or reported
descriptively is decided at run time from the fraction of observations at a
bound (tolerance 5%).

The comparison tests whether the association is specific to behavioural
fragmentation (IV; Van Someren et al., 1996; Xu et al., 2023 *J Gerontol A*)
or shared with rhythm regularity (IS) and strength (RA).

Collinearity: RA correlates -0.886 with L5 by construction and -0.555 with
IV, so the three dimensions are not independent tests. The comparison shows
which dimension carries the association most strongly.

## Designs reported

For each outcome (PHQ-9 depression in adults 18+, Cog_composite_z in 60+) and
each exposure $X \in \{\mathrm{IV}, \mathrm{IS}, \mathrm{RA}\}$:

- **M1 (Total, $c$-path)**: outcome\_z $\sim X_z + $ core covariates.
- **M3 (Direct, $c'$-path)**: outcome\_z $\sim X_z + $ KDM-BA + log SII + core covariates.

$X$, outcome, and (in M3) the mediators are z-standardised on the per-outcome
sample using survey-weighted SDs. Survey-weighted WLS with HC3 errors, no
complex-survey design correction; inference follows the Mplus models.

## Inputs

- `data/processed/8_nhanes_analytic.csv`

## Outputs

- `result/supplementary/SuppTableS9_RAR_Dimension_Specificity.csv` and `.xlsx`
- `result/raw/script12_RAR_full.csv`
- `figure/supplementary/SuppFigS2_RAR_Dimensions_Forest.pdf` + `.png`
- `figure/source_data/Source_Data_SuppFigS2_RAR_Dimensions_Forest.xlsx`

## Step 0 — Imports and paths

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
for q in (R_SUPP, R_RAW, F_SUPP, F_SRC):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
import population; importlib.reload(population)
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

from wls_utils import wls_fit
from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import PHQ9_COLOR, COG_COLOR, GREY, EXP_ORDER
from nature_palette import fmt_ci as _fmt_ci, age_label


## Step 1 — Load + rename + derive PHQ9 / COG_Z

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
    'obj_mvpa_min':   'OBJ_MVPA',
    'KDM_BA_advance': 'KDM_BA_ADV',
    'SII_log':        'SII_LOG',
}
df = df.rename(columns=rename)
# Stored once by script 8; asserted here rather than re-derived inline.
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= ADULT_MIN_AGE].copy()   # the analysis population (population.py)

# PHQ-9 total (canonical)
dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

# Cognition composite z (within-60+ z-mean)
elig_60 = df['AGE'] >= OLD_MIN_AGE
for src, dst in [('memory_score', 'MEM_Z'),
                 ('fluency_score', 'FLUEN_Z'),
                 ('speed_score',  'SPEED_Z')]:
    raw = df[src]
    s = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

print(f'Adults {ADULT_MIN_AGE}+: n = {len(df):,}')
for c in ['RA', 'IS', 'IV']:  # EXP_ORDER; defined later in this notebook
    s = df[c].dropna()
    print(f'  {c:3s} non-null = {len(s):5,}   mean = {s.mean():7.4f}   '
          f'median = {s.median():7.4f}   sd = {s.std():7.4f}')

# Censoring diagnostic for relative amplitude.
v = df.dropna(subset=['RA'])
n_ceil = int((v['RA'] >= 0.9999).sum())
n_l5z  = int((v['L5'] <= 0).sum()) if 'L5' in v.columns else -1
print()
print('Relative amplitude censoring')
print(f'  RA at ceiling (= 1.000) : {n_ceil:,} / {len(v):,} ({100*n_ceil/len(v):.1f}%)')
if n_l5z >= 0:
    print(f'  L5 exactly zero         : {n_l5z:,} / {len(v):,} ({100*n_l5z/len(v):.1f}%)')
print(f'  RA median               : {v["RA"].median():.4f}   (maximum = {v["RA"].max():.4f})')
print('     The classification of each dimension is decided from these numbers at run time below.')

## Step 2 — Helpers (per-exposure sample + survey-weighted z + WLS-HC3)

In [ ]:
M1_COVS    = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
# Volume-adjusted variants of every path live in the Mplus models (the _vol
# series) and their supplementary tables; this notebook does not refit them.
MED_VARS   = ['KDM_BA_ADV', 'SII_LOG']

# IV and IS are computed on a common hourly epoch in
# scripts/5_accelerometer_RAR.ipynb; RA at minute resolution from M10 and L5.
# Whether a dimension is an inferential exposure or descriptive only is decided
# at run time from the fraction of observations piled up at a bound.
CENSOR_TOL   = 0.05    # a dimension is treated as censored if more than 5% of
                       # observations pile up at a bound

def censoring_fraction(col):
    v = df[col].dropna()
    if len(v) == 0:
        return 1.0
    lo, hi = v.min(), v.max()
    at_lo = float((v <= lo + 1e-9).mean())
    at_hi = float((v >= hi - 1e-9).mean())
    return max(at_lo, at_hi)

EXPOSURES = list(EXP_ORDER)
_frac       = {c: censoring_fraction(c) for c in EXPOSURES}
INFERENTIAL = [c for c in EXPOSURES if _frac[c] <= CENSOR_TOL]
CENSORED    = [c for c in EXPOSURES if _frac[c] >  CENSOR_TOL]

print('Bound-pile-up check (fraction of observations at the extreme value):')
for c in EXPOSURES:
    print(f'  {c:3s} {_frac[c]*100:6.2f}%   -> '
          f'{"inferential" if c in INFERENTIAL else "CENSORED, descriptive only"}')
if CENSORED:
    print(f'  NOTE: {CENSORED} exceeded the {CENSOR_TOL*100:.0f}% tolerance and are '
          f'reported descriptively.')
else:
    print('  No dimension is censored; all three are inferential exposures.')

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def weighted_mean_sd(x, w):
    w_tot = np.sum(w)
    mu = np.sum(x * w) / w_tot
    var = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

def per_exposure_sample(exposure_col, outcome_col, extras=(), age_min=ADULT_MIN_AGE):
    needed = [exposure_col, outcome_col, 'WTMEC4YR'] + M1_COVS + list(extras)
    mask = (df['AGE'] >= age_min) & df[needed].notna().all(axis=1)
    return df.loc[mask].copy()

def run_path_stdyx(exposure_col, outcome_col, extras=(), age_min=ADULT_MIN_AGE):
    """WLS-HC3 of outcome_z on exposure_z + covariates (+ extras).
    Exposure, outcome, and mediator extras are z-standardised on the sample.
    """
    ana = per_exposure_sample(exposure_col, outcome_col, extras, age_min=age_min)
    w   = ana['WTMEC4YR'].values
    mu_x, sd_x = weighted_mean_sd(ana[exposure_col].values, w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values,  w)
    x_z = (ana[exposure_col].values - mu_x) / sd_x
    y_z = (ana[outcome_col].values  - mu_y) / sd_y
    cols = [x_z]
    for extra in extras:
        if extra in MED_VARS:
            mu_m, sd_m = weighted_mean_sd(ana[extra].values, w)
            cols.append((ana[extra].values - mu_m) / sd_m)
        else:
            cols.append(ana[extra].values)
    for c in M1_COVS:
        cols.append(ana[c].values)
    X = np.column_stack(cols)
    fit = wls_fit(X, y_z, w=w, se='hc3', add_const=True)
    return {
        'exposure':    exposure_col,
        'outcome':     outcome_col,
        'extras':      ', '.join(extras) if extras else '(none)',
        'n':           int(fit['n']),
        'k':           int(fit['k']),
        'sd_w_x':      sd_x,
        'sd_w_y':      sd_y,
        'beta':        float(fit['params'][1]),
        'se':          float(fit['se'][1]),
        't':           float(fit['t'][1]),
        'p':           float(fit['p'][1]),
        'ci_lo':       float(fit['ci_lo'][1]),
        'ci_hi':       float(fit['ci_hi'][1]),
        'r2_model':    float(fit['r2']),
    }

## Step 3 — Run M1 (Total) and M3 (Direct) for each (exposure × outcome)

In [ ]:
results = []
for outcome_label, outcome_col, age_min in [
    ('Depression (PHQ-9 total)',           'PHQ9',  ADULT_MIN_AGE),
    ('Cognitive function composite (\u226560)', 'COG_Z', OLD_MIN_AGE),
]:
    for exp in EXPOSURES:
        for model_label, extras in [
            ('M1 (Total, c-path)',          ()),
            ('M3 (Direct, c′-path)',   MED_VARS),
        ]:
            r = run_path_stdyx(exp, outcome_col, extras=extras, age_min=age_min)
            r['outcome_label'] = outcome_label
            r['model_label']   = model_label
            r['inference']     = ('inferential' if exp in INFERENTIAL
                                  else 'descriptive (censored exposure)')
            results.append(r)
            print(f"{outcome_label:38s}  {exp}  {model_label:24s}  "
                  f"n={r['n']:,}  beta={r['beta']:+.3f}  "
                  f"CI=({r['ci_lo']:+.3f}, {r['ci_hi']:+.3f})  p={fmt_p(r['p'])}")

results_raw = pd.DataFrame(results)
results_raw.to_csv(R_RAW / 'script12_RAR_full.csv', index=False)
print(f'\nWrote {R_RAW / "script12_RAR_full.csv"}')

## Step 4 — Build Supp Table S9

In [ ]:
rows = []
for outcome_label in ['Depression (PHQ-9 total)',
                      'Cognitive function composite (\u226560)']:
    block = results_raw[results_raw['outcome_label'] == outcome_label].reset_index(drop=True)
    rows.append({'Outcome': outcome_label, 'Exposure': '', 'Model': '',
                 'n': '', 'beta_STDYX (95% CI)': '', 'p': '', 'R2': ''})
    for exp in EXPOSURES:
        sub = block[block['exposure'] == exp].reset_index(drop=True)
        tag = '' if exp in INFERENTIAL else '  [censored - descriptive only]'
        rows.append({'Outcome': '', 'Exposure': exp + tag, 'Model': '',
                     'n': '', 'beta_STDYX (95% CI)': '', 'p': '', 'R2': ''})
        for _, r in sub.iterrows():
            rows.append({'Outcome': '', 'Exposure': '', 'Model': r['model_label'],
                         'n': f"{int(r['n']):,}",
                         'beta_STDYX (95% CI)': fmt_ci(r['beta'], r['ci_lo'], r['ci_hi']),
                         'p': fmt_p(r['p']),
                         'R2': f"{r['r2_model']:.4f}"})

tableS9 = pd.DataFrame(rows)
print(tableS9.to_string(index=False))
tableS9.to_csv(R_SUPP / 'SuppTableS9_RAR_Dimension_Specificity.csv', index=False)

# The classification notes are derived from the run-time check, not typed.
NOTES = [
    'Survey-weighted WLS with HC3 errors; NO complex-survey design correction. '
    'Inference follows the Mplus models; this table is a dimension-specificity '
    'robustness check.',
    'Exposure, outcome and mediator z-standardised with survey-weighted mean '
    'and s.d. on each analytic sample (STDYX analogue); covariates raw.',
    'M1 (Total, c-path): exposure_z + core covariates.',
    "M3 (Direct, c'-path): exposure_z + KDM-BA advancement + log SII + core covariates.",
    'Core covariates: age, sex, BMI, education, wear time, sleep duration.',
    'IV and IS are computed on a common hourly epoch; RA at minute resolution '
    'from M10 and L5.',
]
for c in EXPOSURES:
    frac = _frac[c] * 100
    if c in INFERENTIAL:
        NOTES.append(f'{c}: {frac:.2f}% of observations at a bound -- treated as '
                     'a full inferential exposure.')
    else:
        NOTES.append(f'{c}: {frac:.2f}% of observations at a bound (tolerance '
                     f'{CENSOR_TOL*100:.0f}%) -- reported descriptively only.')
NOTES.append('The classification above is computed at run time from the data; '
             'it is not a fixed list.')

with pd.ExcelWriter(R_SUPP / 'SuppTableS9_RAR_Dimension_Specificity.xlsx',
                    engine='openpyxl') as xw:
    tableS9.to_excel(xw, sheet_name='Supp Table S9', index=False)
    pd.DataFrame({'Notes': NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f"\nwrote {R_SUPP / 'SuppTableS9_RAR_Dimension_Specificity.xlsx'}")

# Move outputs under the previous table number to _superseded/.
old = R_SUPP / 'SuppTableS4_RAR_Dimension_Specificity.xlsx'
oldc = R_SUPP / 'SuppTableS4_RAR_Dimension_Specificity.csv'
for stale in (old, oldc):
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')


## Step 5 — Supp Figure S2 forest plot (3 exposures × 2 outcomes × 2 models)

In [ ]:
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.9,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    'axes.spines.top': False, 'axes.spines.right': False,
})
MM = 1 / 25.4
from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import EXP_COL, EXP_ORDER  # one source of truth
from nature_palette import fmt_ci as _fmt_ci, age_label

fig, axes = plt.subplots(1, 2, figsize=(183 * MM, 66 * MM), sharex=False)
src_sheets = {}
for ax, outcome_label, title, pcol in [
        (axes[0], 'Depression (PHQ-9 total)', f'Depressive symptoms (\u2265{ADULT_MIN_AGE} years)', PHQ9_COLOR),
        (axes[1], 'Cognitive function composite (\u226560)', f'Observed cognitive composite (\u2265{OLD_MIN_AGE} years)', COG_COLOR)]:
    block = results_raw[results_raw['outcome_label'] == outcome_label]
    items = []
    for exp in EXPOSURES:
        for ml, mk in [('M1 (Total, c-path)', 'o'), ("M3 (Direct, c\u2032-path)", 'D')]:
            r = block[(block['exposure'] == exp) & (block['model_label'] == ml)]
            if len(r):
                items.append((exp, ml, mk, r.iloc[0]))
    yt, yl = [], []
    for y, (exp, ml, mk, r) in enumerate(items):
        sig = r['p'] < 0.05
        col = EXP_COL[exp]      # hue = metric, always; fill = significance
        ax.errorbar(r['beta'], -y,
                    xerr=[[r['beta'] - r['ci_lo']], [r['ci_hi'] - r['beta']]],
                    fmt=mk, ms=3.6, color=col, mfc=col if sig else 'white',
                    ecolor=col, elinewidth=0.9, capsize=1.8, mew=0.9)
        yt.append(-y)
        yl.append(f"{exp}  {'total' if ml.startswith('M1') else 'direct'}")
    ax.axvline(0, color='0.4', lw=0.5, ls=(0, (3, 2)))
    ax.set_yticks(yt); ax.set_yticklabels(yl)
    ax.set_title(title, color=pcol, pad=3)
    ax.set_xlabel('Standardized coefficient \u03b2 (95% CI)')
    src_sheets[outcome_label] = block[['exposure', 'model_label', 'n', 'beta',
                                       'ci_lo', 'ci_hi', 'p', 'r2_model']]
fig.tight_layout()
for ax, letter in zip(axes, 'ab'):
    fig.text(ax.get_position().x0 - 0.06, 0.98, letter, ha='right', va='top',
             fontsize=10, fontweight='bold')

stem = 'SuppFigS2_RAR_Dimensions_Forest'
fig.savefig(F_SUPP / f'{stem}.pdf', bbox_inches='tight')
fig.savefig(F_SUPP / f'{stem}.png', bbox_inches='tight', dpi=600)
plt.close(fig)
print(f'wrote {F_SUPP / stem}.pdf + .png')

cap = ['Supplementary Fig. S2 | Total and direct associations of each '
       'rest-activity rhythm metric with depressive symptoms and cognitive function.',
       f'a, depressive symptoms (PHQ-9), adults aged \u2265{ADULT_MIN_AGE} years; b, observed '
       f'cognitive composite, adults aged \u2265{OLD_MIN_AGE} years.',
       'Circles: total association (rhythm metric and core covariates). Diamonds: '
       'direct association, additionally adjusted for KDM biological-age '
       'advancement and log SII. Points are standardized coefficients from '
       'survey-weighted least-squares models with HC3 standard errors; bars are '
       '95% confidence intervals. Open markers indicate P \u2265 0.05 (not adjusted '
       'for multiple comparisons).',
       'Each model used its own available-case sample and weighted standardization, '
       'so the total and direct coefficients are not on a common sample or scale '
       'and their difference does not by itself quantify the change on adding the '
       'biological measures; sample sizes are in Supplementary Table 9.',
       'No complex-survey design correction; supportive analysis, inference '
       'follows the structural models. PHQ-9, 9-item Patient Health '
       'Questionnaire; SII, systemic immune-inflammation index.']
with pd.ExcelWriter(F_SRC / f'Source_Data_{stem}.xlsx', engine='openpyxl') as xw:
    for lab, sub in src_sheets.items():
        sheet = 'PHQ-9' if 'Depression' in lab else 'Cognition'
        sub.to_excel(xw, sheet_name=sheet, index=False)
    pd.DataFrame({'Notes': cap}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SRC / ("Source_Data_" + stem + ".xlsx")}')

# Move superseded raster exports and their source data to _superseded/.
for stale in [F_SUPP / f'{stem}.tiff', F_SRC / 'Source_Data_SuppFigS2.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')
